

# Khaoula Healthy Insurance Data Platform
## Project Setup & Environment Inspection

This notebook performs the initial environment inspection for the
Khaoula Healthy Insurance Data Platform.

The project will demonstrate an Azure Databricks healthcare insurance
data platform using:

- Unity Catalog
- Delta Lake
- PySpark
- SQL
- Lakeflow Spark Declarative Pipelines
- Lakeflow Jobs
- Git-based development

### Data Sources

1. Kaggle healthcare insurance claims dataset
2. SMART Health IT FHIR R4 API
3. Generated operational data

### DP-750 Areas

- Compute
- Unity Catalog
- Data ingestion
- Data modeling
- Data quality
- Data pipelines
- Lakeflow Jobs
- Development lifecycle
- Governance and security
- Monitoring and optimization

In [0]:
# Inspect Spark and Databricks runtime information

print("Spark version:", spark.version)

## 2. Inspect Current Unity Catalog Context

Identify the catalog and schema currently associated with this notebook.


In [0]:
# Identify the current catalog and schema

current_catalog = spark.catalog.currentCatalog()
current_schema = spark.catalog.currentDatabase()

print("Current catalog:", current_catalog)
print("Current schema:", current_schema)

## 3. Inspect Available Catalogs

Review the catalogs currently available to the Databricks user.

In [0]:
# List catalogs visible to the current user

catalogs = spark.sql("SHOW CATALOGS")

display(catalogs)

## 4. Inspect Current Catalog Configuration

Inspect the metadata associated with the current catalog, including
its storage configuration.

In [0]:
# Inspect extended metadata for the current catalog

catalog_details = spark.sql(
    f"DESCRIBE CATALOG EXTENDED `{current_catalog}`"
)

display(catalog_details)

## 5. Identify the Catalog Storage Root

Extract the storage root associated with the current catalog.

This is important because Unity Catalog managed tables require
a managed storage location.

In [0]:
# Extract the storage root from the current catalog metadata

storage_root_row = (
    catalog_details
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()
)

if storage_root_row:
    storage_root = storage_root_row["info_value"]
else:
    storage_root = None

print("Storage root:", storage_root)

## 6. Inspect Available External Locations

Review the Unity Catalog external locations available to the current user.

External locations provide governed access to cloud storage
through Unity Catalog.

In [0]:
%sql
SHOW EXTERNAL LOCATIONS;

In [0]:
%sql
LIST 'abfss://unity-catalog-storage@dbstorageimo3san2ax6n2.dfs.core.windows.net/7405611693577244/';

In [0]:
%sql
DESCRIBE EXTERNAL LOCATION `adb_dp750`;

In [0]:
%sql
--SHOW STORAGE CREDENTIALS;
DESCRIBE STORAGE CREDENTIAL adb_dp750

#CREATING THE PROJECT CARTALOG AND SCHEMAS

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS health_insurance
MANAGED LOCATION 'abfss://unity-catalog-storage@dbstorageimo3san2ax6n2.dfs.core.windows.net/7405611693577244/health_insurance'
COMMENT 'Khaoula Healthy Insurance Data Platform';

CREATE SCHEMA IF NOT EXISTS health_insurance.bronze
COMMENT 'Raw and minimally transformed source data';

CREATE SCHEMA IF NOT EXISTS health_insurance.silver
COMMENT 'Cleaned, validated and conformed data';

CREATE SCHEMA IF NOT EXISTS health_insurance.gold
COMMENT 'Business-ready analytical data';

## 7. Unity Catalog Project Structure

The project uses the following Unity Catalog hierarchy:

health_insurance
├── bronze
├── silver
└── gold

The `health_insurance` catalog uses a dedicated managed storage
location under the existing Databricks Unity Catalog storage root.

Bronze:
- Raw and minimally transformed source data.

Silver:
- Cleaned, validated and conformed data.

Gold:
- Business-ready analytical datasets.